# 05 · Power analysis

**Questions:**

1. What is the smallest effect this experiment could reliably detect for each metric (the **minimum detectable effect**, MDE)?
2. How many customers would an experiment need to detect a given effect?
3. How much does covariate adjustment (notebook 04) change that?

**Method:** the standard normal-approximation formulas for a two-sided test at level $\alpha$ with power $1-\beta$:

$$\text{MDE} = (z_{1-\alpha/2} + z_{1-\beta})\;\sigma\,\sqrt{\tfrac{1}{n_t} + \tfrac{1}{n_c}}, \qquad n_{\text{per arm}} = 2\,\frac{(z_{1-\alpha/2} + z_{1-\beta})^2\,\sigma^2}{\text{MDE}^2}.$$

Defaults: $\alpha = 0.05$, power 80%. $\sigma$ is the control arm's standard deviation ($\sqrt{p(1-p)}$ for rates), the usual choice when planning, because the treated arm's variance is unknown in advance. The functions are checked against `statsmodels` in the tests.

Not used: simulation-based power. With ~21,000 customers per arm, notebook 03 showed the normal approximation holds even for spend (bootstrap and Welch intervals agree to half a cent), so simulation would add complexity without changing the answer.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from hillstrom.data import load_customers
from hillstrom.effects import diff_means_welch
from hillstrom.plots import ARM_COLORS, ARM_LABELS, TEXT_PRIMARY, TEXT_SECONDARY, apply_style, save_figure
from hillstrom.power import mde_means, required_n_means

apply_style()
df = load_customers()
n_arm = df["arm"].value_counts()
control = df[df["arm"] == "control"]
METRICS = ["visit", "conversion", "spend"]
SCALE = {"visit": (100, "pp"), "conversion": (100, "pp"), "spend": (1, "$")}

## 1. What could this experiment detect?

MDE for an email vs No E-Mail comparison at the actual arm sizes (~21,300 each), next to the observed effects from notebook 03. The second MDE column uses $\alpha = 0.05/9$: under the Holm correction across 9 tests, the smallest p-value is compared with 0.05/9, so this is the MDE in the worst case for that correction.

In [2]:
mde_rows = []
for metric in METRICS:
    sd = control[metric].std()
    scale, unit = SCALE[metric]
    row = {
        "metric": metric,
        "unit": unit,
        "control_sd": sd,
        "MDE (alpha 0.05)": scale * mde_means(sd, n_arm["mens"], n_arm["control"]),
        "MDE (alpha 0.05/9)": scale * mde_means(sd, n_arm["mens"], n_arm["control"], alpha=0.05 / 9),
    }
    for email in ["mens", "womens"]:
        effect = df.loc[df["arm"] == email, metric].mean() - control[metric].mean()
        row[f"observed: {ARM_LABELS[email]}"] = scale * effect
    mde_rows.append(row)
mde_table = pd.DataFrame(mde_rows)
mde_table.style.format({c: "{:.3f}" for c in mde_table.columns if c not in ("metric", "unit")}).hide(axis="index")

metric,unit,control_sd,MDE (alpha 0.05),MDE (alpha 0.05/9),observed: Mens E-Mail,observed: Womens E-Mail
visit,pp,0.308,0.836,1.079,7.659,4.523
conversion,pp,0.075,0.205,0.264,0.681,0.311
spend,$,11.588,0.315,0.406,0.770,0.424


Both emails' effects are well above the MDE on every metric, even at the stricter $\alpha$: the experiment was comfortably powered for "did each email work?". The Womens conversion effect (+0.31 pp) is the closest call, at a little above the MDE.

The MDE uses the control arm's spread. The email arms' spend is more spread out (standard deviation about \$15-18 vs \$11.6), so the real MDE for spend is somewhat larger than shown, which matters for the next comparison.